In [ ]:
import os
import glob
from tqdm import tqdm
import re
import math
import pickle
from itertools import islice
import pandas as pd 

In [ ]:
targets=[['smell', 'fmell', 'smelle', 'fmelle']]
spans=['1600-1699', '1700-1799', '1800-1899', '1900-1999']
top_n = 50

In [ ]:
for index, target in enumerate(targets):
  for i,t in enumerate(target):
    t = t.strip()
    target[i] = t  
  targets[index] = target

In [ ]:
all_lines = []

columns_dict = {
# "Numb":0,
"Book":0,
"Smell_Word":1,
"Smell_Source":2,
"Quality":3,
"Odour_Carrier":4,
"Evoked_Odorant":5,
"Location":6,
"Perceiver":7,
"Time":8,
"Circumstances":9,
"Effect":10,
"SentenceBefore":11,
"Sentence":12,
"SentenceAfter":13,
"year":14
}


In [ ]:
totalDict = dict()
for s in spans:
  totalDict[s] = dict()

freqDict = dict()
for s in spans:
  freqDict[s] = dict()

coocDict = dict()
for s in spans:
  coocDict[s] = dict()

In [ ]:
def take(n, iterable):
    """Return the first n items of the iterable as a list."""
    return list(islice(iterable, n))

In [ ]:
def add_to_dict(mydict, span, tokens):
  for t in tokens:
    tmptoken = t.split("_____")[0]
    # if tmptoken.isalpha():
    if len(tmptoken) < 2:
      continue
    if t not in mydict[span]:
      mydict[span][t] = 0
    mydict[span][t]  += 1 

In [ ]:
def intersection(lst1, lst2):
    lst3 = [value for value in lst1 if value in lst2]
    return lst3
count = 0

In [ ]:
with open('/Users/teresapaccosi/Desktop/VARIE/TASTE_IJCL/df_new_all_lemma.tsv','r') as file:
    for line in file:
      line = line.strip("\n")
      parts = line.split("\t")
      # year = int(parts[columns_dict['year']])
      year_value = parts[columns_dict['year']]
      # if year_value:
      #   year = int(year_value)
      try:
        year = int(year_value)
      except ValueError:
        count=count+1
        continue

      for span in spans:
        yStart = int(span.split("-")[0])
        yEnd = int(span.split("-")[1])
        if year > yStart and year <= yEnd:

          tmp_string = re.sub('[^A-Za-z0-9]', " ", parts[columns_dict['Smell_Word']])
          tmp_string = re.sub(" +", " ", tmp_string)
          tmp_string = tmp_string.lower()
          mylist1 = tmp_string.split(" ")
          mylist1 = set(mylist1)
          add_to_dict(totalDict, span, ['total'])                
          add_to_dict(freqDict, span, mylist1)                

          tmp_string = re.sub('[^A-Za-z0-9]', " ", parts[columns_dict['Quality']])
          tmp_string = re.sub(" +", " ", tmp_string)
          tmp_string = tmp_string.lower()
          mylist2 = tmp_string.split(" ")
          add_to_dict(totalDict, span, ['total'])                
          add_to_dict(freqDict, span, mylist2)                

          coocsList = []
          
          for x in mylist1:
            for y in mylist2:
              if len(x) > 0 and len(y) > 0:
                coocsList.append(str(x).lower()+" "+str(y))
          add_to_dict(coocDict, span, coocsList) 

In [ ]:
with open('totalDict.pkl', 'wb') as file:
    pickle.dump(totalDict, file)
with open('freqDict.pkl', 'wb') as file:
    pickle.dump(freqDict, file)
with open('coocDict.pkl', 'wb') as file:
    pickle.dump(coocDict, file)

In [ ]:
with open('totalDict.pkl', 'rb') as file:
    totalDict = pickle.load(file)
with open('freqDict.pkl', 'rb') as file:
    freqDict = pickle.load(file)
with open('coocDict.pkl', 'rb') as file:
    coocDict = pickle.load(file) 

In [ ]:
for group in targets:
  print(group)
  print()
  for span in spans:
    total_freq = totalDict[span]['total']
    freq_target = 0
    for target in group:
      if target not in freqDict[span]:
        continue
      freq_target = freq_target + freqDict[span][target]
    for p in coocDict[span]:
      freq_pair = coocDict[span][p]
      if freq_pair <3:
        continue
      p_list = p.split(" ")
      if len(intersection(group,p_list)) > 0:
        for w in p_list:
          w = w.lower()
          if w in group:
            continue
          if len(w)<4:
            continue
          if w not in freqDict[span]:
            continue
          freq_cooc = freqDict[span][w]
          
          # print(span,p,w, total_freq, freq_target, freq_cooc, freq_pair)
          print(span, w, freq_target, freq_pair)

In [ ]:
pmiDict = dict()
for s in spans:
    pmiDict[s] = dict()

# treshold:
freq_pair_threshold = 10

for group in targets:
    print(group)
    print()
    for span in spans:
        total_freq = totalDict[span]['total']
        freq_target = 0
        for target in group:
            if target not in freqDict[span]:
                continue
            freq_target = freq_target + freqDict[span][target]
        
        for p in coocDict[span]:
            freq_pair = coocDict[span][p]
            # treshold:
            if freq_pair <= freq_pair_threshold:
                continue
            
            p_list = p.split(" ")
            if len(intersection(group, p_list)) > 0:
                for w in p_list:
                    w = w.lower()
                    if len(w)<4:
                        continue
                    if w in group:
                        continue
                    if w not in freqDict[span]:
                        continue
                    freq_cooc = freqDict[span][w]
                    
                    pxy = freq_pair / total_freq
                    px = freq_target / total_freq
                    py = freq_cooc / total_freq
                    pmi_value = math.log(pxy / (px * py),2)
                    pmiDict[span][w] = pmi_value

        sorted_pmiDict = sorted(pmiDict[span].items(), key=lambda x: x[1], reverse=True)
        converted_dict = dict(sorted_pmiDict)

        n_items = take(top_n, converted_dict.items())
        print(span)
        for x in n_items:
            print(str(x[0]) + "\t" + str(x[1]))

        print()

In [ ]:
import pandas as pd
import re
import math


targets = [['smell', 'fmell', 'smelle', 'fmelle']]
spans = ['1600-1699', '1700-1799', '1800-1899', '1900-1999']
top_n = 50
freq_pair_threshold = 10


targets = [[t.strip() for t in group] for group in targets]


df = pd.read_csv(
    "df_new_all_lemma.tsv",
    sep="\t",
    on_bad_lines="skip"
)

df["year"] = pd.to_numeric(df["year"], errors="coerce")
df = df.dropna(subset=["year"])
df["year"] = df["year"].astype(int)


totalDict = {s: 0 for s in spans}
freqDict = {s: {} for s in spans}
coocDict = {s: {} for s in spans}


def clean(text):
    text = str(text)
    text = re.sub(r"[^A-Za-z0-9]", " ", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text.split()

def add_freq(d, span, tokens):
    for t in tokens:
        if len(t) < 2:
            continue
        d[span][t] = d[span].get(t, 0) + 1

def get_span(year):
    for s in spans:
        a, b = map(int, s.split("-"))
        if a < year <= b:
            return s
    return None

for _, row in df.iterrows():

    year = row["year"]
    span = get_span(year)

    if span is None:
        continue

    smell_tokens = set(clean(row["Smell_Word"]))
    quality_tokens = clean(row["Quality"])

    # total observations
    totalDict[span] += 1

    # unigram counts
    add_freq(freqDict, span, smell_tokens)
    add_freq(freqDict, span, quality_tokens)

    # co-occurrences
    for s in smell_tokens:
        for q in quality_tokens:
            if len(s) < 1 or len(q) < 1:
                continue
            key = f"{s} {q}"
            coocDict[span][key] = coocDict[span].get(key, 0) + 1

for group in targets:

    print("\nTARGET:", group, "\n")

    for span in spans:

        total_freq = totalDict[span]

        freq_target = sum(freqDict[span].get(t, 0) for t in group)

        pair_freqs = {}

        for pair, freq in coocDict[span].items():

            if freq <= freq_pair_threshold:
                continue

            smell, quality = pair.split()

            if smell in group:
                if len(quality) < 4:
                    continue

                pair_freqs[quality] = pair_freqs.get(quality, 0) + freq

        pmi_scores = {}

        for quality, freq_pair in pair_freqs.items():

            freq_cooc = freqDict[span].get(quality, 0)

            if freq_target == 0 or freq_cooc == 0:
                continue

            px = freq_target / total_freq
            py = freq_cooc / total_freq
            pxy = freq_pair / total_freq

            pmi_scores[quality] = math.log2(pxy / (px * py))

        print("SPAN:", span)

        top_words = sorted(
            pmi_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )[:top_n]

        for w, score in top_words:
            print(f"{w}\t{score:.3f}")

        print()

/var/folders/j8/2fw1pn3n4zb1y5l8gt8s56dc0000gn/T/ipykernel_54777/1931758247.py:15: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(



TARGET: ['smell', 'fmell', 'smelle', 'fmelle'] 

SPAN: 1600-1699
excel	1.137
nettle	1.137
fmelling	1.063
delightsome	1.055
ranck	1.050
pleasantly	1.030
ranke	0.989
vnpleasant	0.989
rank	0.978
terebinthinate	0.931
manifest	0.931
rancke	0.931
earthy	0.910
lovely	0.906
musty	0.903
asweet	0.903
crude	0.900
goatish	0.896
unusual	0.896
rammish	0.886
rankly	0.880
sweetly	0.877
somewhat	0.873
offensiue	0.857
sant	0.848
altogether	0.815
ungrateful	0.798
unwholsome	0.789
cordial	0.789
narcotick	0.781
fragrantly	0.770
strongly	0.730
oyly	0.722
pretty	0.722
farre	0.722
unpleasant	0.716
pleafant	0.704
savoury	0.690
delightfull	0.683
fulsome	0.678
well	0.674
ftrong	0.673
comfortable	0.632
mixt	0.630
gross	0.630
link	0.617
smoaky	0.597
strong	0.590
cool	0.585
lothsome	0.583

SPAN: 1700-1799
bituminous	1.308
wooingly	1.287
fmelling	1.287
phofphoreal	1.287
acute	1.232
mortify	1.204
fant	1.194
acutely	1.187
unpleafant	1.186
daverous	1.180
fubacrid	1.176
pretty	1.171
fomewhat	1.161
somewhat	1.155
link	1